# Traditional Methods (PSD_MSD_ACF_FORMA)

## Overview
This section defines a shared data pipeline and experiment configuration so that PSD, MSD, ACF, and FORMA are evaluated under identical conditions.

### What this section does
- Loads trajectory samples and ground-truth parameters (`k`, `gamma`, `m`, `T`, `fs`, `tau_int`).
- Optionally truncates the dataset to a user-defined sample count.
- Provides reusable helpers for window extraction, scalar access, and binned MRE visualization.

### Key configuration
- `data_path`: Path to the input dataset file.
- `window_size`: Number of points used from each trajectory for estimation.
- `n_bins`: Number of logarithmic bins for MRE statistics.
- `num_samples`: Optional cap on the number of evaluated samples.
- `msd_max_lag` / `msd_fit_min_points` / `msd_eval_samples`: Controls for MSD fitting and evaluation.

### Outputs prepared for later sections
- `positions`: Trajectory matrix with shape `N x L`.
- `k_true`, `gamma_true`: Ground-truth labels used for error evaluation.
- `fs`, `m`, `T`, `tau_all`: Per-sample or global physical/sampling metadata.

### Reproducibility notes
- This section does not perform inversion by itself.
- It standardizes inputs and utilities for all downstream traditional methods.
- Keeping this setup centralized makes benchmarking easier to reproduce in open-source workflows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, optimize, constants
from tqdm import tqdm
from types import SimpleNamespace
import os
import time

cfg = SimpleNamespace(
    data_path='Test_data1e4_ALL.npz',
    window_size=100,
    n_bins=50,
    num_samples=None,
    msd_max_lag=30000,
    msd_fit_min_points=40,
    msd_eval_samples=None,
)

if not os.path.exists(cfg.data_path):
    raise FileNotFoundError(f'Data file not found: {cfg.data_path}')

data = np.load(cfg.data_path, allow_pickle=True)
positions = data['position']
fs = data['fs']
k_true = data['k0']
gamma_true = data['gamma']
m = data['m']
T = data['T']
tau_all = data['tau_int'] if 'tau_int' in data.files else None

if positions.ndim == 1:
    positions = positions[None, :]

if getattr(cfg, 'num_samples', None) is not None:
    n_keep = min(cfg.num_samples, positions.shape[0])
    positions = positions[:n_keep]
    k_true = k_true[:n_keep] if np.size(k_true) > 1 else k_true
    gamma_true = gamma_true[:n_keep] if np.size(gamma_true) > 1 else gamma_true
    m = m[:n_keep] if np.size(m) > 1 else m
    T = T[:n_keep] if np.size(T) > 1 else T
    fs = fs[:n_keep] if np.size(fs) > 1 else fs
    if tau_all is not None:
        tau_all = tau_all[:n_keep] if np.size(tau_all) > 1 else tau_all

print(f'Data loaded: {positions.shape[0]} samples, window_size={cfg.window_size}')


def extract_window(x, window_size, mode='center'):
    x = np.asarray(x).ravel()
    n = x.size
    if window_size is None or n == window_size:
        return x
    if n > window_size:
        start = (n - window_size) // 2 if mode == 'center' else 0
        return x[start:start + window_size]
    pad = window_size - n
    left = pad // 2
    right = pad - left
    return np.pad(x, (left, right), mode='reflect')


def _scalar(arr, idx, fallback=None):
    if arr is None:
        return fallback
    arr = np.asarray(arr)
    if arr.ndim == 0:
        return float(arr)
    return float(arr[idx])


def plot_mre_vs_pred_bins(pred, true, title, n_bins=50):
    pred = np.asarray(pred)
    true = np.asarray(true)
    valid = np.isfinite(pred) & np.isfinite(true) & (pred > 0) & (true > 0)
    if np.sum(valid) < 10:
        print(f'{title}: Not enough valid samples, skipping binned plot.')
        return

    pred_v = pred[valid]
    true_v = true[valid]
    mre_v = np.abs(pred_v - true_v) / true_v

    log_true = np.log10(true_v)
    edges = np.linspace(np.min(log_true), np.max(log_true), n_bins + 1)
    centers = 10 ** ((edges[:-1] + edges[1:]) / 2)

    mean_mre = np.full(n_bins, np.nan)
    med_mre = np.full(n_bins, np.nan)
    p25 = np.full(n_bins, np.nan)
    p75 = np.full(n_bins, np.nan)

    for i in range(n_bins):
        in_bin = (log_true >= edges[i]) & (log_true < edges[i + 1] if i < n_bins - 1 else log_true <= edges[i + 1])
        if np.any(in_bin):
            vals = mre_v[in_bin]
            mean_mre[i] = np.mean(vals)
            med_mre[i] = np.median(vals)
            p25[i] = np.percentile(vals, 25)
            p75[i] = np.percentile(vals, 75)

    show = np.isfinite(mean_mre)
    if np.sum(show) == 0:
        print(f'{title}: No valid statistics after binning.')
        return

    plt.figure(figsize=(8, 5))
    plt.xscale('log')
    plt.plot(centers[show], mean_mre[show] * 100, 'o-', ms=4, lw=1.4, label='Mean MRE')
    plt.plot(centers[show], med_mre[show] * 100, 's-', ms=3.5, lw=1.2, label='Median MRE')
    plt.fill_between(centers[show], p25[show] * 100, p75[show] * 100, alpha=0.2, label='IQR (25%-75%)')
    plt.xlabel('True parameter value')
    plt.ylabel('MRE (%)')
    plt.title(title.replace('Predicted', 'True'))
    plt.grid(True, which='both', ls='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 1) PSD Method
This section estimates `k` and `gamma` from the power spectral density using a physically grounded aliased PSD model.

### Method summary
- Computes Welch PSD from each trajectory window.
- Fits a theoretical PSD model in log-domain with bounded nonlinear least squares.
- Supports finite exposure via `tau_i` and optional alias summation through `alias_K`.

### Inputs and outputs
- Input: one trajectory, `fs`, `m`, `T`, and optional `tau_i`.
- Output: `k_hat`, `gamma_hat` for each sample.
- Evaluation: relative error and binned MRE plots against ground truth.

In [ ]:
def psd_theory_aliased(freqs, fs_i, tau_i, k, gamma, m_i, T_i, alias_K):
    w = 2 * np.pi * freqs
    base = 2 * (2 * constants.Boltzmann * T_i * gamma) / ((k - m_i * w**2) ** 2 + (gamma * w) ** 2)
    if tau_i is not None and tau_i > 0:
        base *= np.sinc(freqs * tau_i) ** 2
    if alias_K <= 0:
        return base

    n_vec = np.arange(-alias_K, alias_K + 1)
    psd = np.zeros_like(freqs, dtype=float)
    for n in n_vec:
        shift = freqs + n * fs_i
        w_shift = 2 * np.pi * shift
        term = (2 * constants.Boltzmann * T_i * gamma) / ((k - m_i * w_shift**2) ** 2 + (gamma * w_shift) ** 2)
        if tau_i is not None and tau_i > 0:
            term *= np.sinc(shift * tau_i) ** 2
        psd += term
    return psd


def fit_psd(position, fs_i, m_i, T_i, tau_i=None, window_size=None, alias_K=30, fmax_ratio=0.45, use_logfit=True):
    x = extract_window(position, window_size, mode='center')
    x = x - np.mean(x)
    if len(x) < 32:
        return np.nan, np.nan

    nperseg = min(max(256, len(x) // 8), len(x))
    freq, psd_emp = signal.welch(x, fs_i, window='hann', nperseg=nperseg, scaling='density')
    valid = (freq > 0) & (freq < fmax_ratio * fs_i) & np.isfinite(psd_emp) & (psd_emp > 0)
    if np.sum(valid) < 16:
        return np.nan, np.nan
    freq = freq[valid]
    psd_emp = psd_emp[valid]

    var = np.var(x)
    k_init = max(constants.Boltzmann * T_i / max(var, 1e-30), 1e-9)
    omega0 = np.sqrt(k_init / m_i)
    gamma_init = max(m_i * omega0 / 5.0, 1e-13)
    noise_floor = np.median(psd_emp[-max(len(psd_emp) // 10, 5):])

    p0 = np.array([np.log10(k_init), np.log10(gamma_init), np.log10(max(noise_floor, 1e-24))], dtype=float)
    bounds = (np.array([-9.0, -12.0, -30.0]), np.array([-2.0, -5.0, -12.0]))

    def model(freqs, log_k, log_gamma, log_N0):
        k = 10.0 ** log_k
        gamma = 10.0 ** log_gamma
        N0 = 10.0 ** log_N0
        psd = psd_theory_aliased(freqs, fs_i, tau_i, k, gamma, m_i, T_i, alias_K)
        return psd + N0

    eps = 1e-300

    def residuals(p):
        pred = model(freq, *p)
        if use_logfit:
            return np.log(np.clip(pred, eps, None)) - np.log(np.clip(psd_emp, eps, None))
        return pred - psd_emp

    try:
        res = optimize.least_squares(residuals, p0, bounds=bounds, max_nfev=50000)
    except Exception:
        return np.nan, np.nan

    if not res.success:
        return np.nan, np.nan

    k_hat = 10.0 ** res.x[0]
    gamma_hat = 10.0 ** res.x[1]
    if np.isfinite(k_hat) and np.isfinite(gamma_hat) and k_hat > 0 and gamma_hat > 0:
        return k_hat, gamma_hat
    return np.nan, np.nan


alias_K = 0
fmax_ratio = 0.45
use_logfit = True

N_total = positions.shape[0]
k_pred_psd = np.full(N_total, np.nan)
gamma_pred_psd = np.full(N_total, np.nan)

print(f'Running precise PSD analysis on all {N_total} samples...')
psd_window_times_ms = []
for i in tqdm(range(N_total), desc='PSD analysis'):
    fs_i = _scalar(fs, i, float(fs))
    m_i = _scalar(m, i, m)
    T_i = _scalar(T, i, T)
    tau_i = _scalar(tau_all, i, None)
    if tau_i is None or not np.isfinite(tau_i) or tau_i <= 0:
        tau_i = 1.0 / fs_i
    t0 = time.perf_counter()

    k_hat, gamma_hat = fit_psd(
        positions[i], fs_i, m_i, T_i,
        tau_i=tau_i,
        window_size=cfg.window_size,
        alias_K=alias_K,
        fmax_ratio=fmax_ratio,
        use_logfit=use_logfit,
    )
    k_pred_psd[i] = k_hat
    gamma_pred_psd[i] = gamma_hat
    psd_window_times_ms.append((time.perf_counter() - t0) * 1000.0)

if len(psd_window_times_ms) > 0:
    t_arr = np.asarray(psd_window_times_ms, dtype=float)
    print(f"PSD timing: mean={np.mean(t_arr):.4f} ms/window, median={np.median(t_arr):.4f}, p95={np.percentile(t_arr, 95):.4f}, total={np.sum(t_arr)/1000.0:.3f} s")

valid_idx = np.isfinite(k_pred_psd) & np.isfinite(gamma_pred_psd) & (k_pred_psd > 0) & (gamma_pred_psd > 0)
print(f'PSD method - valid predictions: {np.sum(valid_idx)}/{N_total}')
if np.any(valid_idx):
    k_rel_error = np.mean(np.abs((k_pred_psd[valid_idx] - k_true[valid_idx]) / k_true[valid_idx]))
    g_rel_error = np.mean(np.abs((gamma_pred_psd[valid_idx] - gamma_true[valid_idx]) / gamma_true[valid_idx]))
    print(f'k relative error: {k_rel_error * 100:.2f}%')
    print(f'gamma relative error: {g_rel_error * 100:.2f}%')

    plot_mre_vs_pred_bins(k_pred_psd[valid_idx], k_true[valid_idx], 'PSD: MRE vs Predicted k (log-domain 50 bins)', n_bins=cfg.n_bins)
    plot_mre_vs_pred_bins(gamma_pred_psd[valid_idx], gamma_true[valid_idx], 'PSD: MRE vs Predicted gamma (log-domain 50 bins)', n_bins=cfg.n_bins)

## 2) MSD Method
This section fits a robust mean-squared displacement model to estimate `k` and `gamma` under noisy and finite-length trajectories.

### Method summary
- Builds autocorrelation with FFT acceleration, then derives MSD over logarithmic lag samples.
- Uses a damped-oscillator MSD model with additive offset and robust loss (`soft_l1`).
- Applies multi-start optimization to reduce local-minimum sensitivity.

### Inputs and outputs
- Input: one trajectory, `fs`, `m`, `T`, and MSD fitting controls.
- Output: `k_hat`, `gamma_hat` for each evaluated sample.
- Evaluation: relative error and binned MRE plots against ground truth.

In [ ]:
def msd_method(position, fs_i, m_i, T_i, max_lag=300, window_size=None, min_fit_points=40):
    dt = 1.0 / fs_i
    sig = extract_window(position, window_size, mode='center') if window_size else np.asarray(position)
    x = np.asarray(sig, dtype=np.float64).ravel()
    x = x - np.mean(x)
    n = x.size
    if n < 128:
        return np.nan, np.nan

    max_lag_eff = int(min(max_lag, n // 3))
    if max_lag_eff < 40:
        return np.nan, np.nan

    nfft = 1 << ((2 * n - 1).bit_length())
    fx = np.fft.rfft(x, nfft)
    acf = np.fft.irfft(fx * np.conjugate(fx), nfft)[:n]
    acf /= np.arange(n, 0, -1)

    lags = np.unique(np.ceil(np.logspace(0, np.log10(max_lag_eff), num=220)).astype(int))
    lags = lags[(lags > 0) & (lags < n)]
    if lags.size < min_fit_points:
        return np.nan, np.nan

    var0 = max(acf[0], 1e-30)
    msd = 2.0 * (var0 - acf[lags])
    tau = lags * dt

    valid = np.isfinite(msd) & np.isfinite(tau) & (tau > 0)
    msd = msd[valid]
    tau = tau[valid]
    if msd.size < min_fit_points:
        return np.nan, np.nan

    msd = np.maximum(msd, 1e-30)

    kb = constants.Boltzmann

    def msd_model(t, log_k, log_beta, log_c):
        k = 10.0 ** log_k
        beta = 10.0 ** log_beta
        c = 10.0 ** log_c
        omega0 = np.sqrt(k / m_i)
        half = beta / 2.0
        disc = half**2 - omega0**2
        if disc < 0:
            omega_d = np.sqrt(-disc)
            env = np.exp(-half * t)
            core = 2.0 * kb * T_i / k * (1.0 - env * (np.cos(omega_d * t) + (half / omega_d) * np.sin(omega_d * t)))
        else:
            sqrt_disc = np.sqrt(disc)
            r1 = half + sqrt_disc
            r2 = half - sqrt_disc
            C = (r2 * np.exp(-r1 * t) - r1 * np.exp(-r2 * t)) / (r2 - r1 + 1e-30)
            core = 2.0 * kb * T_i / k * (1.0 - C)
        return core + c

    tail_n = max(10, msd.size // 5)
    plateau = float(np.median(msd[-tail_n:]))
    if not np.isfinite(plateau) or plateau <= 0:
        plateau = float(np.percentile(msd, 80))
    if not np.isfinite(plateau) or plateau <= 0:
        return np.nan, np.nan

    k_est = np.clip(2.0 * kb * T_i / plateau, 1e-9, 1e-2)

    idx_10 = np.where(msd >= 0.1 * plateau)[0]
    idx_80 = np.where(msd >= 0.8 * plateau)[0]
    tau_min_fit = tau[idx_10[0]] if idx_10.size > 0 else tau[min(2, tau.size - 1)]
    tau_max_fit = tau[idx_80[0]] if idx_80.size > 0 else tau[int(0.7 * (tau.size - 1))]
    if tau_max_fit <= tau_min_fit:
        tau_min_fit = tau[min(2, tau.size - 1)]
        tau_max_fit = tau[int(0.8 * (tau.size - 1))]

    fit_mask = (tau >= tau_min_fit) & (tau <= tau_max_fit)
    if np.sum(fit_mask) < min_fit_points:
        fit_mask = np.ones_like(tau, dtype=bool)

    tau_fit = tau[fit_mask]
    msd_fit = msd[fit_mask]
    if tau_fit.size < min_fit_points:
        return np.nan, np.nan

    idx_e = np.where(msd_fit >= plateau * (1.0 - 1.0 / np.e))[0]
    tau_e = tau_fit[idx_e[0]] if idx_e.size > 0 else np.median(tau_fit)
    beta_est = np.clip(1.0 / max(tau_e, 1e-8), 1.0, 1e8)

    c_est = np.clip(np.percentile(msd_fit[:max(3, msd_fit.size // 8)], 30), 1e-24, plateau)

    p0_base = np.array([np.log10(k_est), np.log10(beta_est), np.log10(c_est)], dtype=float)
    bounds = (
        np.array([-10.0, 0.0, -30.0], dtype=float),
        np.array([-2.0, 9.0, -6.0], dtype=float),
    )

    eps = 1e-300

    def residuals(p):
        pred = msd_model(tau_fit, *p)
        return np.log(np.clip(pred, eps, None)) - np.log(np.clip(msd_fit, eps, None))

    starts = [
        p0_base,
        p0_base + np.array([0.2, -0.3, 0.0]),
        p0_base + np.array([-0.2, 0.3, -0.3]),
        p0_base + np.array([0.0, 0.5, 0.3]),
    ]

    best = None
    best_cost = np.inf
    for p0 in starts:
        p0 = np.clip(p0, bounds[0] + 1e-8, bounds[1] - 1e-8)
        try:
            res = optimize.least_squares(
                residuals,
                p0,
                bounds=bounds,
                loss='soft_l1',
                f_scale=0.3,
                max_nfev=30000,
            )
        except Exception:
            continue
        if res.success and res.cost < best_cost:
            best = res
            best_cost = res.cost

    if best is None:
        return np.nan, np.nan

    k_fit = 10.0 ** best.x[0]
    beta_fit = 10.0 ** best.x[1]
    gamma_fit = beta_fit * m_i

    if np.isfinite(k_fit) and np.isfinite(gamma_fit) and k_fit > 0 and gamma_fit > 0:
        return float(k_fit), float(gamma_fit)
    return np.nan, np.nan


n_eval = len(positions) if cfg.msd_eval_samples is None else min(int(cfg.msd_eval_samples), len(positions))
print(f'Running MSD analysis on {n_eval}/{len(positions)} samples...')
k_pred_msd = np.full(n_eval, np.nan)
gamma_pred_msd = np.full(n_eval, np.nan)
msd_window_times_ms = []

for i in tqdm(range(n_eval), desc='MSD analysis'):
    fs_i = _scalar(fs, i, float(fs))
    m_i = _scalar(m, i, m)
    T_i = _scalar(T, i, T)
    t0 = time.perf_counter()
    kf, gf = msd_method(
        positions[i],
        fs_i,
        m_i,
        T_i,
        max_lag=cfg.msd_max_lag,
        window_size=cfg.window_size,
        min_fit_points=cfg.msd_fit_min_points,
    )
    k_pred_msd[i] = kf
    gamma_pred_msd[i] = gf
    msd_window_times_ms.append((time.perf_counter() - t0) * 1000.0)

if len(msd_window_times_ms) > 0:
    t_arr = np.asarray(msd_window_times_ms, dtype=float)
    print(f"MSD timing: mean={np.mean(t_arr):.4f} ms/window, median={np.median(t_arr):.4f}, p95={np.percentile(t_arr, 95):.4f}, total={np.sum(t_arr)/1000.0:.3f} s")

k_true_eval = k_true[:n_eval] if np.size(k_true) > 1 else np.full(n_eval, float(k_true))
g_true_eval = gamma_true[:n_eval] if np.size(gamma_true) > 1 else np.full(n_eval, float(gamma_true))

valid_idx = np.isfinite(k_pred_msd) & np.isfinite(gamma_pred_msd) & (k_pred_msd > 0) & (gamma_pred_msd > 0)
print(f'MSD method - valid predictions: {np.sum(valid_idx)}/{n_eval}')
if np.any(valid_idx):
    k_rel_error = np.mean(np.abs(k_pred_msd[valid_idx] - k_true_eval[valid_idx]) / k_true_eval[valid_idx])
    g_rel_error = np.mean(np.abs(gamma_pred_msd[valid_idx] - g_true_eval[valid_idx]) / g_true_eval[valid_idx])
    print(f'k relative error: {k_rel_error * 100:.2f}%')
    print(f'gamma relative error: {g_rel_error * 100:.2f}%')

    plot_mre_vs_pred_bins(k_pred_msd[valid_idx], k_true_eval[valid_idx], 'MSD: MRE vs Predicted k (log-domain 50 bins)', n_bins=cfg.n_bins)
    plot_mre_vs_pred_bins(gamma_pred_msd[valid_idx], g_true_eval[valid_idx], 'MSD: MRE vs Predicted gamma (log-domain 50 bins)', n_bins=cfg.n_bins)

## 3) ACF Method
This section estimates `k` and `gamma` by fitting the normalized autocorrelation function in the time domain.

### Method summary
- Computes normalized ACF from each trajectory window.
- Fits an analytical damped-oscillator ACF form with weighted nonlinear least squares.
- Converts fitted dynamic parameters (`omega0`, `beta`) into physical parameters (`k`, `gamma`).

### Inputs and outputs
- Input: one trajectory, `fs`, and `m`.
- Output: `k_hat`, `gamma_hat` for each sample.
- Evaluation: relative error and binned MRE plots against ground truth.

In [ ]:
def acf_method_robust(position, fs_i, m_i, window_size=None):
    dt = 1.0 / fs_i
    sig = extract_window(position, window_size, mode='center')
    x = sig - np.mean(sig)
    n = len(x)

    max_lag = window_size // 4 if window_size else n // 4
    if max_lag < 20:
        return np.nan, np.nan

    acf_full = signal.fftconvolve(x, x[::-1], mode='full')
    acf = acf_full[n - 1:n - 1 + max_lag]
    acf /= np.arange(n, n - max_lag, -1)
    acf = acf / (acf[0] + 1e-30)

    def acf_model(t, omega0, beta):
        half = beta / 2.0
        disc = half**2 - omega0**2
        if disc < 0:
            omega_d = np.sqrt(-disc)
            return np.exp(-half * t) * (np.cos(omega_d * t) + (half / omega_d) * np.sin(omega_d * t))
        sqrt_disc = np.sqrt(disc)
        r1 = half + sqrt_disc
        r2 = half - sqrt_disc
        return (r2 * np.exp(-r1 * t) - r1 * np.exp(-r2 * t)) / (r2 - r1 + 1e-30)

    t = np.arange(max_lag) * dt
    t_fit = t
    y_fit = acf
    if len(t_fit) < 10:
        return np.nan, np.nan

    f_psd, pxx = signal.welch(x, fs_i, nperseg=min(n, 256))
    f_peak = f_psd[np.argmax(pxx)] if len(pxx) > 1 else fs_i / 10
    omega0_est = 2 * np.pi * max(f_peak, 1.0)

    idx_e = np.where(y_fit <= 1 / np.e)[0]
    tau_e = t_fit[idx_e[0]] if len(idx_e) > 0 else t_fit[-1] / 2
    beta_est = 1.0 / max(tau_e, 1e-6)

    p0 = [omega0_est, beta_est]
    bounds = ([1.0, 1.0], [np.pi * fs_i, 1e8])
    sigma = 1.0 / np.sqrt(np.arange(n, n - max_lag, -1))

    try:
        popt, _ = optimize.curve_fit(
            acf_model,
            t_fit,
            y_fit,
            p0=p0,
            bounds=bounds,
            sigma=sigma,
            absolute_sigma=False,
            maxfev=2000,
        )
        omega0, beta = popt
        k = m_i * omega0**2
        gamma = beta * m_i
        if k > 0 and gamma > 0:
            return k, gamma
    except Exception:
        pass

    return np.nan, np.nan


k_pred_acf = np.full(len(positions), np.nan)
gamma_pred_acf = np.full(len(positions), np.nan)
print(f'Running robust ACF analysis on all {len(positions)} samples...')
acf_window_times_ms = []

for i in tqdm(range(len(positions)), desc='ACF analysis'):
    fs_i = _scalar(fs, i, float(fs))
    t0 = time.perf_counter()
    kf, gf = acf_method_robust(positions[i], fs_i, m[i], window_size=cfg.window_size)
    k_pred_acf[i] = kf
    gamma_pred_acf[i] = gf
    acf_window_times_ms.append((time.perf_counter() - t0) * 1000.0)

if len(acf_window_times_ms) > 0:
    t_arr = np.asarray(acf_window_times_ms, dtype=float)
    print(f"ACF timing: mean={np.mean(t_arr):.4f} ms/window, median={np.median(t_arr):.4f}, p95={np.percentile(t_arr, 95):.4f}, total={np.sum(t_arr)/1000.0:.3f} s")

valid_idx = np.isfinite(k_pred_acf) & np.isfinite(gamma_pred_acf)
print(f'ACF method - valid predictions: {np.sum(valid_idx)}/{len(positions)}')
if np.any(valid_idx):
    k_rel_error = np.mean(np.abs(k_pred_acf[valid_idx] - k_true[valid_idx]) / k_true[valid_idx])
    g_rel_error = np.mean(np.abs(gamma_pred_acf[valid_idx] - gamma_true[valid_idx]) / gamma_true[valid_idx])
    print(f'k relative error: {k_rel_error * 100:.2f}%')
    print(f'gamma relative error: {g_rel_error * 100:.2f}%')

    plot_mre_vs_pred_bins(k_pred_acf[valid_idx], k_true[valid_idx], 'ACF: MRE vs Predicted k (log-domain 50 bins)', n_bins=cfg.n_bins)
    plot_mre_vs_pred_bins(gamma_pred_acf[valid_idx], gamma_true[valid_idx], 'ACF: MRE vs Predicted gamma (log-domain 50 bins)', n_bins=cfg.n_bins)

## 4) FORMA Method
This section performs frequency-domain fitting with an analytical underdamped Langevin PSD form to recover `k` and `gamma`.

### Method summary
- Computes periodogram-based PSD from each trajectory window.
- Fits an analytical underdamped Langevin PSD with an additive noise floor.
- Uses bounded nonlinear least squares in log-domain for stable parameter recovery.

### Inputs and outputs
- Input: one trajectory, `fs`, `m`, and `T`.
- Output: `k_hat`, `gamma_hat` for each sample.
- Evaluation: relative error and binned MRE plots against ground truth.

In [ ]:
def forma_method(position, fs_i, m_i, T_i, window_size=None):
    kb = constants.Boltzmann
    sig = extract_window(position, window_size, mode='center')
    x = sig - np.mean(sig)
    if len(x) < 64:
        return np.nan, np.nan

    freqs, psd_emp = signal.periodogram(x, fs_i, scaling='density', window='hann')
    valid = (freqs > 0) & np.isfinite(psd_emp) & (psd_emp > 0)
    if np.sum(valid) < 16:
        return np.nan, np.nan

    freqs = freqs[valid]
    omega = 2 * np.pi * freqs
    psd_emp = psd_emp[valid]

    var = np.var(x)
    k_init = np.clip(kb * T_i / max(var, 1e-30), 1e-9, 1e-2)
    omega0_init = np.sqrt(k_init / m_i)
    gamma_init = np.clip(m_i * omega0_init / 5.0, 1e-12, 1e-6)
    noise_init = np.median(psd_emp[-max(psd_emp.size // 10, 4):])

    p0 = np.log10([k_init, gamma_init, max(noise_init, 1e-24)])
    bounds = (np.array([-9.0, -12.0, -30.0]), np.array([-2.0, -6.0, -12.0]))

    def model_psd(log_k, log_gamma, log_N0):
        k = 10.0 ** log_k
        gamma = 10.0 ** log_gamma
        n0 = 10.0 ** log_N0
        theory = (2.2 * kb * T_i * gamma) / ((k - m_i * omega**2) ** 2 + (gamma * omega) ** 2)
        return theory + n0

    eps = 1e-300

    def residuals(p):
        pred = model_psd(*p)
        return np.log(np.clip(pred, eps, None)) - np.log(np.clip(psd_emp, eps, None))

    res = optimize.least_squares(residuals, p0, bounds=bounds, max_nfev=20000)
    if not res.success:
        return np.nan, np.nan

    k_fit = 10.0 ** res.x[0]
    gamma_fit = 10.0 ** res.x[1]

    if np.isfinite(k_fit) and np.isfinite(gamma_fit) and k_fit > 0 and gamma_fit > 0:
        return k_fit, gamma_fit
    return np.nan, np.nan


print(f'Running FORMA analysis on all {len(positions)} samples...')
k_pred_forma = np.full(len(positions), np.nan)
gamma_pred_forma = np.full(len(positions), np.nan)
forma_window_times_ms = []

for i in tqdm(range(len(positions)), desc='FORMA analysis'):
    fs_i = _scalar(fs, i, float(fs))
    t0 = time.perf_counter()
    kf, gf = forma_method(positions[i], fs_i, m[i], T[i], window_size=cfg.window_size)
    k_pred_forma[i] = kf
    gamma_pred_forma[i] = gf
    forma_window_times_ms.append((time.perf_counter() - t0) * 1000.0)

if len(forma_window_times_ms) > 0:
    t_arr = np.asarray(forma_window_times_ms, dtype=float)
    print(f"FORMA timing: mean={np.mean(t_arr):.4f} ms/window, median={np.median(t_arr):.4f}, p95={np.percentile(t_arr, 95):.4f}, total={np.sum(t_arr)/1000.0:.3f} s")

valid_idx = np.isfinite(k_pred_forma) & np.isfinite(gamma_pred_forma)
print(f'FORMA method - valid predictions: {np.sum(valid_idx)}/{len(positions)}')
if np.any(valid_idx):
    k_rel_error = np.mean(np.abs(k_pred_forma[valid_idx] - k_true[valid_idx]) / k_true[valid_idx])
    g_rel_error = np.mean(np.abs(gamma_pred_forma[valid_idx] - gamma_true[valid_idx]) / gamma_true[valid_idx])
    print(f'k relative error: {k_rel_error * 100:.2f}%')
    print(f'gamma relative error: {g_rel_error * 100:.2f}%')

    plot_mre_vs_pred_bins(k_pred_forma[valid_idx], k_true[valid_idx], 'FORMA: MRE vs Predicted k (log-domain 50 bins)', n_bins=cfg.n_bins)
    plot_mre_vs_pred_bins(gamma_pred_forma[valid_idx], gamma_true[valid_idx], 'FORMA: MRE vs Predicted gamma (log-domain 50 bins)', n_bins=cfg.n_bins)